```
Artist ──< Album ──< Track >── Genre
                       │
Custo|mer ──< Invoice ──< InvoiceLine
    │
Employee (SupportRepId 로 담당 직원 연결)
```

In [4]:
from helper import chinook_conn, q, show_tables, show_columns

conn = chinook_conn()
print(show_tables(conn))
show_columns(conn, 'Album')

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


,name,type
0,AlbumId,INTEGER
1,Title,NVARCHAR(160)
2,ArtistId,INTEGER


# INNER JOIN

In [8]:
sql = '''
SELECT al.ArtistId, al.Title, ar.Name
FROM Album AS al
    JOIN Artist AS ar ON al.ArtistId = ar.ArtistId
LIMIT 5
'''
q(conn, sql)

,ArtistId,Title,Name
0,1,For Those About To Rock We Salute You,AC/DC
1,2,Balls to the Wall,Accept
2,2,Restless and Wild,Accept
3,1,Let There Be Rock,AC/DC
4,3,Big Ones,Aerosmith


In [ ]:
# Track -> AlbumId
# Album -> AlbumId, ArtistId
# Artist -> ArtistId
# select: 트랙이름, 앨범의 타이틀, 아티스트 이름

In [12]:
sql = '''
SELECT t.Name as track, 
    al.Title as album,
    ar.Name as artist
FROM Track t
    JOIN Album al ON t.AlbumId = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
LIMIT  3
'''
q(conn, sql)

,track,album,artist
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC
1,Balls to the Wall,Balls to the Wall,Accept
2,Fast As a Shark,Restless and Wild,Accept


In [20]:
sql = '''
SELECT COUNT(*) as n
FROM artist ar
    LEFT JOIN Album al ON ar.ArtistId = al.ArtistId
'''
sql = '''
SELECT ar.ArtistId, al.ArtistId, ar.Name --, al.Title
FROM Artist ar
    LEFT JOIN Album al ON ar.ArtistId = al.ArtistId
WHERE al.AlbumId IS NULL
LIMIT 5
'''
q(conn, sql)

,ArtistId,ArtistId,Name
0,25,None,Milton Nascimento & Bebeto
1,26,None,Azymuth
2,28,None,João Gilberto
3,29,None,Bebel Gilberto
4,30,None,Jorge Vercilo


# JOIN + GROUP BY

In [22]:
sql = '''
SELECT ar.Name as artist, count(*) as n_track
FROM Track t
    JOIN Album al ON t.AlbumId = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
GROUP BY ar.Name
ORDER BY n_track DESC
'''
q(conn, sql)

,artist,n_track
0,Iron Maiden,213
1,U2,135
2,Led Zeppelin,114
3,Metallica,112
4,Lost,92
...,...,...
199,"Academy of St. Martin in the Fields, Sir Nevil...",1
200,"Academy of St. Martin in the Fields, John Birc...",1
201,Academy of St. Martin in the Fields Chamber En...,1
202,Aaron Goldberg,1


In [ ]:
# 고객 (Customer), 주문(Invoice)
# Invoice.Total <- 요게 매출
# 국가별, 매출순위

In [25]:
sql = '''
SELECT c.Country, ROUND(SUM(i.Total), 2) as revenue
FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY 1  -- c.Country
ORDER BY 2 DESC  -- SUM(i.Total)
LIMIT 10
'''
q(conn, sql)
# 11:30 분까지 휴식

,Country,revenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86
6,Czech Republic,90.24
7,Portugal,77.24
8,India,75.26
9,Chile,46.62


# SubQuery

In [30]:
sql = '''
SELECT InvoiceId, CustomerId, Total
FROM Invoice
WHERE Total >= (SELECT AVG(Total) FROM Invoice)
ORDER BY Total DESC
'''
# sql = '''
# SELECT AVG(Total) FROM Invoice
# '''
q(conn, sql)

,AVG(Total)
0,5.651942


In [32]:
sql = '''
SELECT FirstName, LastName, Country
From Customer
WHERE CustomerId IN (SELECT CustomerId FROM Invoice WHERE total > 20)
'''
# sql = '''
# SELECT CustomerId FROM Invoice WHERE total > 20
# '''
q(conn, sql)

,FirstName,LastName,Country
0,Helena,Holý,Czech Republic
1,Richard,Cunningham,USA
2,Ladislav,Kovács,Hungary
3,Hugh,O'Reilly,Ireland


In [34]:
sql = '''
WITH country_sales AS (
    SELECT c.Country, ROUND(SUM(i.Total), 2) as revenue
    FROM Customer c
        JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY 1
)
SELECT * FROM country_sales
WHERE revenue > 100
ORDER BY revenue DESC
'''
q(conn, sql)

,Country,revenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86


# Window Function 

In [41]:
sql = ''' 
SELECT c.Country, i.Total, 
    ROUND(AVG(i.Total) OVER (PARTITION BY c.Country), 2) as country_avg
FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
'''
sql = '''
SELECT c.Country, AVG(i.Total) 
FROM Customer c 
    JOIN Invoice i ON c.CustomerId = i.CustomerId
Group By 1
'''
q(conn, sql)

,Country,AVG(i.Total)
0,Argentina,5.374286
1,Australia,5.374286
2,Austria,6.088571
3,Belgium,5.374286
4,Brazil,5.431429
5,Canada,5.427857
6,Chile,6.660000
7,Czech Republic,6.445714
8,Denmark,5.374286
9,Finland,5.945714


In [43]:
sql = '''
SELECT Total, 
    ROW_NUMBER() OVER (ORDER BY Total DESC) as row_num,
    RANK()       OVER (ORDER BY Total DESC) as rnk,
    DENSE_RANK() OVER (ORDER BY Total DESC) as dense
FROM Invoice
LIMIT 6
'''
q(conn, sql)
# 14:00 까지 휴식

,Total,row_num,rnk,dense
0,25.86,1,1,1
1,23.86,2,2,2
2,21.86,3,3,3
3,21.86,4,3,3
4,18.86,5,5,4
5,18.86,6,5,4
